---
title: Long, hold, and short classification
execute:
  enabled: true
---

This example builds a three-class model from bundled SPY daily bars, following the full research sequence: get data, split chronologically, label future returns, train, validate, and test. The resulting `short`, `hold`, and `long` probabilities feed the multiclass ROC and precision-recall plots.

The model is intentionally small and the results are realistic rather than optimized. Treat the workflow as an evaluation template, not as a trading strategy.

In [1]:
import numpy as np
import pandas as pd
import plotly.io as pio
from scipy.optimize import minimize

import qrt as q

pio.renderers.default = "notebook_connected"

## 1. Get data

Load the bundled SPY OHLCV history, then construct features known at the close of each session. The target is the return over the next five sessions, which is kept separate from the feature table.

In [2]:
spy = q.data.datasets.load("spy")
returns = spy["close"].pct_change()
horizon = 5

features = pd.DataFrame(
    {
        "return_1d": returns,
        "momentum_5d": spy["close"].pct_change(5),
        "momentum_20d": spy["close"].pct_change(20),
        "volatility_20d": returns.rolling(20).std(),
        "range_1d": (spy["high"] - spy["low"]) / spy["close"],
        "volume_change_5d": spy["volume"].pct_change(5),
    }
).replace([np.inf, -np.inf], np.nan).dropna()
forward_return = spy["close"].shift(-horizon).div(spy["close"]).sub(1)

features.tail()

,return_1d,momentum_5d,momentum_20d,volatility_20d,range_1d,volume_change_5d
datetime,,,,,,
2026-07-14,0.003551,0.005510,0.016201,0.008817,0.006225,-0.196206
2026-07-15,0.003964,0.012624,0.002550,0.007932,0.007128,0.001768
2026-07-16,-0.005419,-0.001317,0.003097,0.007911,0.008911,0.119882
2026-07-17,-0.009897,-0.015445,0.005729,0.007712,0.008731,0.484927
2026-07-20,-0.001614,-0.009450,-0.006227,0.007344,0.009729,0.144103


## 2. Split chronologically

Use the oldest 60% of observations for training, the next 20% for validation, and the newest 20% for testing. Five sessions are purged before each boundary so no label's forward-return window crosses into the following partition.

In [3]:
train_boundary = int(len(features) * 0.60)
validation_boundary = int(len(features) * 0.80)

X = {
    "train": features.iloc[: train_boundary - horizon],
    "validation": features.iloc[train_boundary : validation_boundary - horizon],
    "test": features.iloc[validation_boundary:-horizon],
}

pd.DataFrame(
    {
        name: {
            "start": frame.index.min().date(),
            "end": frame.index.max().date(),
            "observations": len(frame),
        }
        for name, frame in X.items()
    }
).T

,start,end,observations
train,2000-02-01,2015-12-07,3988
validation,2015-12-15,2021-03-23,1326
test,2021-03-31,2026-07-13,1326


## 3. Label future returns

Fit the action thresholds on training returns only. The bottom 30% become `short`, the top 30% become `long`, and the middle 40% become `hold`. Reusing those fixed cutoffs on validation and test preserves genuine distribution shifts.

In [4]:
classes = ["short", "hold", "long"]
short_cutoff, long_cutoff = forward_return.loc[X["train"].index].quantile([0.30, 0.70])


def make_labels(index):
    future = forward_return.loc[index]
    labels = np.select(
        [future <= short_cutoff, future >= long_cutoff],
        ["short", "long"],
        default="hold",
    )
    return pd.Series(labels, index=index, name="position")


y = {name: make_labels(frame.index) for name, frame in X.items()}

pd.DataFrame(
    {name: labels.value_counts(normalize=True) for name, labels in y.items()}
).T.loc[:, classes].style.format("{:.1%}")

position,short,hold,long
train,30.0%,40.0%,30.0%
validation,21.0%,48.5%,30.5%
test,25.9%,42.5%,31.6%


## 4. Train

Standardize with training statistics and fit a multinomial logistic regression. The implementation uses SciPy's optimizer so the tutorial remains offline and does not require an additional machine-learning dependency.

In [5]:
train_mean = X["train"].mean()
train_std = X["train"].std().replace(0, 1)
X_scaled = {name: (frame - train_mean) / train_std for name, frame in X.items()}


def softmax(scores):
    shifted = scores - scores.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def fit_multinomial_logit(feature_frame, labels, l2_penalty):
    design = np.column_stack([np.ones(len(feature_frame)), feature_frame.to_numpy()])
    target = pd.Categorical(labels, categories=classes).codes

    def objective(flat_weights):
        weights = flat_weights.reshape(design.shape[1], len(classes))
        probabilities = softmax(design @ weights)
        loss = -np.log(
            np.clip(probabilities[np.arange(len(target)), target], 1e-12, 1)
        ).mean()
        loss += 0.5 * l2_penalty * np.square(weights[1:]).sum()

        residual = probabilities.copy()
        residual[np.arange(len(target)), target] -= 1
        gradient = design.T @ residual / len(target)
        gradient[1:] += l2_penalty * weights[1:]
        return loss, gradient.ravel()

    result = minimize(
        objective,
        np.zeros(design.shape[1] * len(classes)),
        jac=True,
        method="L-BFGS-B",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x.reshape(design.shape[1], len(classes))


def predict_proba(feature_frame, weights):
    design = np.column_stack([np.ones(len(feature_frame)), feature_frame.to_numpy()])
    return pd.DataFrame(
        softmax(design @ weights),
        index=feature_frame.index,
        columns=classes,
    )

## 5. Validate

Fit a small regularization grid and select the penalty with the lowest validation log loss. The test partition remains untouched during this choice.

In [6]:
def log_loss(labels, probabilities):
    target = pd.Categorical(labels, categories=classes).codes
    selected = probabilities.to_numpy()[np.arange(len(target)), target]
    return -np.log(np.clip(selected, 1e-12, 1)).mean()


models = {}
validation_rows = []
for l2_penalty in [0.0, 0.001, 0.01, 0.1, 1.0]:
    weights = fit_multinomial_logit(X_scaled["train"], y["train"], l2_penalty)
    validation_score = predict_proba(X_scaled["validation"], weights)
    models[l2_penalty] = weights
    validation_rows.append(
        {
            "l2_penalty": l2_penalty,
            "log_loss": log_loss(y["validation"], validation_score),
        }
    )

validation_results = pd.DataFrame(validation_rows)
best_l2 = validation_results.loc[validation_results["log_loss"].idxmin(), "l2_penalty"]
weights = models[best_l2]
validation_results.style.format({"l2_penalty": "{:.3g}", "log_loss": "{:.3f}"})

,l2_penalty,log_loss
0,0,0.998
1,0.001,0.998
2,0.01,0.999
3,0.1,1.005
4,1,1.032


## 6. Test

Score the held-out test period once with the selected model. `q.stats` returns tidy one-vs-rest curves for every action plus micro and macro averages; no plotting dependency is required to inspect or export these results.

In [7]:
y_true = y["test"]
y_score = predict_proba(X_scaled["test"], weights)

roc_curves = q.stats.multiclass_roc_curve(y_true, y_score)
pr_curves = q.stats.multiclass_precision_recall_curve(y_true, y_score)

roc_summary = roc_curves.assign(
    label=lambda frame: frame["class"].fillna(frame["curve"])
).groupby("label", sort=False)["auc"].first()
pr_summary = pr_curves.assign(
    label=lambda frame: frame["class"].fillna(frame["curve"])
).groupby("label", sort=False)["average_precision"].first()

pd.concat(
    [roc_summary.rename("roc_auc"), pr_summary.rename("average_precision")],
    axis=1,
).style.format("{:.3f}")

,roc_auc,average_precision
label,,
short,0.567,0.307
hold,0.649,0.561
long,0.604,0.408
micro,0.631,0.471
macro,0.607,0.425


## ROC curve

Each solid line treats one action as positive and the other two as negative. AUC measures ranking quality across every decision threshold; the diagonal reference represents random ranking.

In [8]:
q.plot.roc(
    y_true,
    y_score,
    title="SPY five-session position ROC curve",
).show()

## Precision-recall curve

Average precision summarizes precision across recall levels. This view exposes the harder minority-class decisions and should be read against each class's observed test prevalence.

In [9]:
q.plot.precision_recall(
    y_true,
    y_score,
    title="SPY five-session position precision-recall curve",
).show()